# Unit execution for `sz_nagy_dilation.py`

This notebook checks the direct Sz.-Nagy one-ancilla dilation on **manual, tweakable** operators and states.

It demonstrates:
- contraction checking and optional auto-scaling,
- construction of the defect operators,
- unitarity of the dilation,
- state-level postselection,
- density-matrix-level postselection.


In [2]:
import sys
import os
from pathlib import Path

import numpy as np
import scipy.linalg as la

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sz_nagy_dilation import (
    build_sz_nagy_dilation,
    is_contraction,
    operator_norm_2,
    defect_operator_right,
    defect_operator_left,
    apply_sz_nagy_dilation_to_state,
    apply_sz_nagy_dilation_to_density_matrix,
    normalized_density_matrix_from_state,
    partial_trace_ancilla_two_block,
)


## 1. Define a tweakable operator and state

In [4]:
# Example 1: a near-diagonal contraction
K = np.array([
    [0.92, 0.08],
    [0.00, 0.65],
], dtype=complex)

# Try also a non-contraction to test auto-scaling, e.g.
# K = np.array([[1.2, 0.1], [0.0, 0.8]], dtype=complex)

psi = np.array([
    np.sqrt(0.7),
    np.sqrt(0.3) * np.exp(1j * np.pi / 6),
], dtype=complex)
psi = psi / np.linalg.norm(psi)

print('K =')
print(K)
print('state psi =', psi)
print('||K||_2 =', operator_norm_2(K))
print('is contraction =', is_contraction(K))


K =
[[0.92+0.j 0.08+0.j]
 [0.  +0.j 0.65+0.j]]
state psi = [0.83666 +0.j       0.474342+0.273861j]
||K||_2 = 0.9268197260239522
is contraction = True


## 2. Build the Sz.-Nagy dilation

In [5]:
dilation = build_sz_nagy_dilation(K, auto_scale=True)

print('scale_factor =', dilation.scale_factor)
print('scaled operator =')
print(dilation.scaled_operator)
print('defect_right = sqrt(I - K^dag K) =')
print(dilation.defect_right)
print('defect_left = sqrt(I - K K^dag) =')
print(dilation.defect_left)

unitarity_error = la.norm(
    dilation.unitary.conj().T @ dilation.unitary - np.eye(dilation.unitary.shape[0]),
    ord='fro',
)
print('unitarity error (Frobenius) =', unitarity_error)


scale_factor = 1.0
scaled operator =
[[0.92+0.j 0.08+0.j]
 [0.  +0.j 0.65+0.j]]
defect_right = sqrt(I - K^dag K) =
[[ 0.386559+0.j -0.064589+0.j]
 [-0.064589+0.j  0.752946+0.j]]
defect_left = sqrt(I - K K^dag) =
[[ 0.380943+0.j -0.045634+0.j]
 [-0.045634+0.j  0.758563+0.j]]
unitarity error (Frobenius) = 5.63804624838172e-16


## 3. State-level postselection check

In [7]:
state_result = apply_sz_nagy_dilation_to_state(dilation, psi)

print('ancilla-|0> branch =')
print(state_result.ancilla_zero_branch)
print('target K|psi> =')
print(state_result.target_state)
print('state error norm =', state_result.state_error_norm)
print('p_success =', state_result.p_success)
print('p_failure =', state_result.p_failure)
print('p_success + p_failure =', state_result.p_success + state_result.p_failure)

p_success_exact = np.real(np.vdot(psi, dilation.scaled_operator.conj().T @ dilation.scaled_operator @ psi))
print('Exact <psi|K^dag K|psi> =', float(p_success_exact))


ancilla-|0> branch =
[0.807675+0.021909j 0.308322+0.17801j ]
target K|psi> =
[0.807675+0.021909j 0.308322+0.17801j ]
state error norm = 0.0
p_success = 0.7795681889483063
p_failure = 0.2204318110516939
p_success + p_failure = 1.0000000000000002
Exact <psi|K^dag K|psi> = 0.7795681889483062


## 4. Density-matrix-level postselection check

In [8]:
rho = normalized_density_matrix_from_state(psi)
rho_result = apply_sz_nagy_dilation_to_density_matrix(dilation, rho)

print('ancilla-|0><0| block =')
print(rho_result.ancilla_zero_block)
print('Target K rho K^dag =')
print(rho_result.target_density)
print('Frobenius error =', rho_result.density_error_frobenius)
print('p_success =', rho_result.p_success)
print('p_failure =', rho_result.p_failure)
print('trace(output) =', np.trace(rho_result.ancilla_system_output))


ancilla-|0><0| block =
[[0.652818+0.j       0.252924-0.137019j]
 [0.252924+0.137019j 0.12675 +0.j      ]]
Target K rho K^dag =
[[0.652818+0.j       0.252924-0.137019j]
 [0.252924+0.137019j 0.12675 +0.j      ]]
Frobenius error = 0.0
p_success = 0.7795681889483064
p_failure = 0.22043181105169385
trace(output) = (1.0000000000000002-8.673617379884035e-19j)


## 5. Optional: reduced system output after tracing out ancilla

In [9]:
rho_system_reduced = partial_trace_ancilla_two_block(rho_result.ancilla_system_output)
print('Reduced system density after tracing out ancilla =')
print(rho_system_reduced)
print('This equals K rho K^dag + D_K rho D_K.')


Reduced system density after tracing out ancilla =
[[0.738852-0.j       0.338023-0.202753j]
 [0.338023+0.202753j 0.261148+0.j      ]]
This equals K rho K^dag + D_K rho D_K.


## 6. Try a fully manual density matrix

In [10]:
rho_manual = np.array([
    [0.55, 0.14 + 0.08j],
    [0.14 - 0.08j, 0.45],
], dtype=complex)

rho_manual_result = apply_sz_nagy_dilation_to_density_matrix(dilation, rho_manual)
print('manual p_success =', rho_manual_result.p_success)
print('manual Frobenius error =', rho_manual_result.density_error_frobenius)
print('manual ancilla-|0><0| block =')
print(rho_manual_result.ancilla_zero_block)


manual p_success = 0.6791330000000002
manual Frobenius error = 0.0
manual ancilla-|0><0| block =
[[0.489008-0.j      0.10712 +0.04784j]
 [0.10712 -0.04784j 0.190125+0.j     ]]


## 7. Minimal custom template

In [11]:
K_user = np.array([
    [0.80, 0.10],
    [0.00, 0.60],
], dtype=complex)

rho_user = np.array([
    [0.60, 0.20 - 0.05j],
    [0.20 + 0.05j, 0.40],
], dtype=complex)

dil_user = build_sz_nagy_dilation(K_user, auto_scale=True)
res_user = apply_sz_nagy_dilation_to_density_matrix(dil_user, rho_user)

print('scale_factor =', dil_user.scale_factor)
print('p_success =', res_user.p_success)
print('Frobenius error =', res_user.density_error_frobenius)
print('target block match =', np.allclose(res_user.ancilla_zero_block, res_user.target_density))


scale_factor = 1.0
p_success = 0.5640000000000001
Frobenius error = 0.0
target block match = True
